# Step 5 — NammaMart ETL pipeline test

This notebook runs the same ETL implementation used by the Airflow DAG. It is intentionally local and repeatable so the star schema, quarantine, reconciliation and reports can be inspected before starting Docker/Airflow.

Flow: extract → validate → quarantine → transform → load DuckDB → reconcile payments → build reports.

In [ ]:
from pathlib import Path
import sys
import duckdb

# Import the exact module that the Airflow DAG will use.
sys.path.insert(0, str(Path('include').resolve()))
from nammamart_etl import run_etl

# Use an assignment-specific warehouse and quarantine directory so this test
# does not overwrite the original Meridian demo warehouse.
warehouse = Path('assignment/warehouse_samy_step5.duckdb')
quarantine = Path('assignment/quarantine_samy_step5')
result = run_etl(
    warehouse_path=warehouse,
    quarantine_dir=quarantine,
    run_id='step5-local-test',
)

print('ETL run completed')
print(f"Rows extracted:   {result['rows_extracted']:,}")
print(f"Rows loaded:      {result['rows_loaded']:,}")
print(f"Rows quarantined: {result['rows_quarantined']:,}")
print(f"Reconciliation rows: {result['reconciliation_rows']:,}")
print(f"Warehouse: {result['warehouse_path']}")

In [ ]:
# Inspect the tables created by the idempotent DuckDB load.
con = duckdb.connect(str(warehouse), read_only=True)
table_counts = con.execute("""
        SELECT table_name, estimated_size
        FROM duckdb_tables()
        WHERE schema_name = 'main'
        ORDER BY table_name
    """).fetchdf()
display(table_counts)

# The fact-table count is the finance-safe count after critical failures
# have been excluded and dimension joins have enforced referential integrity.
display(con.execute('SELECT COUNT(*) AS fact_rows FROM etl_fact_orders').fetchdf())
display(con.execute('SELECT * FROM etl_daily_zone_revenue ORDER BY order_date, zone, store_id LIMIT 10').fetchdf())
display(con.execute('SELECT * FROM etl_category_margin ORDER BY category').fetchdf())
con.close()

In [ ]:
# Demonstrate idempotency by running the same source data a second time.
second = run_etl(
    warehouse_path=warehouse,
    quarantine_dir=quarantine,
    run_id='step5-local-rerun',
)
con = duckdb.connect(str(warehouse), read_only=True)
fact_count_after_rerun = con.execute('SELECT COUNT(*) FROM etl_fact_orders').fetchone()[0]
con.close()
print(f"Fact rows after rerun: {fact_count_after_rerun:,}")
print('The table is replaced from the same source, so revenue is not double-counted.')